In [1]:
import numpy as np
import pandas as pd

In [2]:
path="/content/drive/MyDrive/mulcam_bigdata/3조 : 최종 프로젝트/공용/AI명령어 데이터/명령어 음성(노인남여)/AI로봇 데이터_50대_training.csv"

In [3]:
pd.read_csv(path)

,대화,성별,연령대,지역,사투리,녹음환경,소음환경,파일이름
0,나 열 시에는 체크인 할 수 있는 거야?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4001-01-02-ANN-F-07-A.wav
1,이거 어떻게 퇴실 해?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4002-01-02-ANN-F-07-A.wav
2,주차요금 나온 거 다 보여 줘.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4003-01-02-ANN-F-07-A.wav
3,주차비를 결제해 주지.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4004-01-02-ANN-F-07-A.wav
4,아무나 간병인 불러 줄래?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4005-01-02-ANN-F-07-A.wav
...,...,...,...,...,...,...,...,...
838307,체크아웃 좀 하고 싶다.,Female,50~59,서울/인천/경기,경상,가정,가정,script1_f_0222-5996-01-02-JHS-F-07-A.wav
838308,병원 푸드코트 어디야?,Female,50~59,서울/인천/경기,경상,가정,가정,script1_f_0222-5997-01-02-JHS-F-07-A.wav
838309,차림표에 나온 음식 가격은 어떻게 되지?,Female,50~59,서울/인천/경기,경상,가정,가정,script1_f_0222-5998-01-02-JHS-F-07-A.wav
838310,출국장 입구 어디인지 찾아봐.,Female,50~59,서울/인천/경기,경상,가정,가정,script1_f_0222-5999-01-02-JHS-F-07-A.wav


In [4]:
raw = pd.read_csv(path)
raw.head()

,대화,성별,연령대,지역,사투리,녹음환경,소음환경,파일이름
0,나 열 시에는 체크인 할 수 있는 거야?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4001-01-02-ANN-F-07-A.wav
1,이거 어떻게 퇴실 해?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4002-01-02-ANN-F-07-A.wav
2,주차요금 나온 거 다 보여 줘.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4003-01-02-ANN-F-07-A.wav
3,주차비를 결제해 주지.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4004-01-02-ANN-F-07-A.wav
4,아무나 간병인 불러 줄래?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4005-01-02-ANN-F-07-A.wav


In [5]:
df = raw.copy()
df.head()

,대화,성별,연령대,지역,사투리,녹음환경,소음환경,파일이름
0,나 열 시에는 체크인 할 수 있는 거야?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4001-01-02-ANN-F-07-A.wav
1,이거 어떻게 퇴실 해?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4002-01-02-ANN-F-07-A.wav
2,주차요금 나온 거 다 보여 줘.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4003-01-02-ANN-F-07-A.wav
3,주차비를 결제해 주지.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4004-01-02-ANN-F-07-A.wav
4,아무나 간병인 불러 줄래?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4005-01-02-ANN-F-07-A.wav


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 838312 entries, 0 to 838311
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   대화      838312 non-null  object
 1   성별      838312 non-null  object
 2   연령대     838312 non-null  object
 3   지역      838312 non-null  object
 4   사투리     838312 non-null  object
 5   녹음환경    838312 non-null  object
 6   소음환경    838312 non-null  object
 7   파일이름    838312 non-null  object
dtypes: object(8)
memory usage: 51.2+ MB


#LDA 분석

In [7]:
!pip install -q pandas numpy scikit-learn konlpy pyLDAvis

In [8]:
import pandas as pd
import numpy as np
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis.lda_model

# 1. 텍스트 전처리
def preprocess_text(text):
    okt = Okt()
    tokens = okt.nouns(text)  # 명사만 추출
    return ' '.join(tokens)

df['preprocessed'] = df['대화'].apply(preprocess_text)

# 2. 단어 벡터화
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix = vectorizer.fit_transform(df['preprocessed'])

# 3. LDA 모델 학습
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_output = lda_model.fit_transform(doc_term_matrix)

In [9]:
# 4. 결과 시각화
pyLDAvis.enable_notebook()
vis = pyLDAvis.lda_model.prepare(lda_model, doc_term_matrix, vectorizer)
pyLDAvis.display(vis)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [10]:
# 5. 토픽별 주요 단어 출력
def print_topics(model, vectorizer, top_n=10):
    for idx, topic in enumerate(model.components_):
        print(f"Topic {idx + 1}:")
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                for i in topic.argsort()[:-top_n - 1:-1]])

print_topics(lda_model, vectorizer)

Topic 1:
[('숙박', 79771.4217530572), ('예약', 67459.08927808264), ('시설', 41783.80972693262), ('확인', 33405.598399779614), ('병원', 23127.491595758613), ('지도', 21506.19632257601), ('체크', 19465.576043940502), ('정보', 15724.101042653743), ('안내', 13438.96475018167), ('부탁', 12812.65477425429)]
Topic 2:
[('주차', 50752.19887154369), ('요금', 41279.1980938568), ('차비', 32329.548415942583), ('지금', 26294.11756480745), ('병원', 22883.764036308043), ('얼마', 22768.528666944792), ('시간', 22329.833638659766), ('내과', 19664.41285545619), ('담당', 15998.182881983294), ('진료', 14467.509684409662)]
Topic 3:
[('예약', 106520.43895671751), ('병원', 52394.059518296664), ('진료', 42720.64370651202), ('시간', 39179.90995854022), ('언제', 19941.49506945047), ('오늘', 18168.308812456093), ('변경', 17113.013027651272), ('취소', 13577.199038818515), ('관계자', 12437.722586103766), ('선생님', 11537.550297626258)]
Topic 4:
[('직원', 27228.654502260582), ('숙소', 25607.408468620233), ('근처', 24556.165346172194), ('시간', 23354.85444767259), ('버스', 21550.573169533

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


결과 표로 만들어보기

#중복 제거 값 LDA 분석

In [11]:
# 중복 제거
df_clean = df.drop_duplicates(subset=["대화"])
df_clean.reset_index(drop=True, inplace=True)
df_clean.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,대화,성별,연령대,지역,사투리,녹음환경,소음환경,파일이름,preprocessed
0,나 열 시에는 체크인 할 수 있는 거야?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4001-01-02-ANN-F-07-A.wav,나 열 시 체크 수
1,이거 어떻게 퇴실 해?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4002-01-02-ANN-F-07-A.wav,거 퇴실 해
2,주차요금 나온 거 다 보여 줘.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4003-01-02-ANN-F-07-A.wav,주차 요금 거
3,주차비를 결제해 주지.,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4004-01-02-ANN-F-07-A.wav,차비 결제 주지
4,아무나 간병인 불러 줄래?,Female,50~59,서울/인천/경기,경기/서울,가정,가정,e_0356-4005-01-02-ANN-F-07-A.wav,아무나 간병인 줄


In [12]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17399 entries, 0 to 17398
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   대화            17399 non-null  object
 1   성별            17399 non-null  object
 2   연령대           17399 non-null  object
 3   지역            17399 non-null  object
 4   사투리           17399 non-null  object
 5   녹음환경          17399 non-null  object
 6   소음환경          17399 non-null  object
 7   파일이름          17399 non-null  object
 8   preprocessed  17399 non-null  object
dtypes: object(9)
memory usage: 1.2+ MB


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [13]:
import pandas as pd
import numpy as np
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
import pyLDAvis.lda_model

# 1. 텍스트 전처리
def preprocess_text(text):
    if pd.isna(text):
        return ""
    okt = Okt()
    tokens = okt.nouns(str(text))  # 문자열로 변환
    return ' '.join(tokens)

# NaN 값 확인
print("전처리 전 NaN 값의 수:", df_clean['대화'].isna().sum())

# 전처리 적용
df_clean['preprocessed'] = df_clean['대화'].apply(preprocess_text)

# 전처리 후 빈 문자열 확인
print("전처리 후 빈 문자열의 수:", (df_clean['preprocessed'] == '').sum())

# 빈 문자열 제거
df_clean = df_clean[df_clean['preprocessed'] != '']

# 2. 단어 벡터화
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix = vectorizer.fit_transform(df_clean['preprocessed'])

# 3. LDA 모델 학습
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_output = lda_model.fit_transform(doc_term_matrix)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


전처리 전 NaN 값의 수: 0


<ipython-input-13-3bd4136b0f94>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['preprocessed'] = df_clean['대화'].apply(preprocess_text)


전처리 후 빈 문자열의 수: 3


In [14]:
# 4. 결과 시각화
pyLDAvis.enable_notebook()
vis = pyLDAvis.lda_model.prepare(lda_model, doc_term_matrix, vectorizer)
pyLDAvis.display(vis)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [15]:
# 5. 토픽별 주요 단어 출력
def print_topics(model, vectorizer, top_n=10):
    feature_names = vectorizer.get_feature_names_out()
    for idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-top_n - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]
        print(f"Topic {idx + 1}:")
        for feat, weight in zip(top_features, weights):
            print(f"{feat}: {weight:.4f}")
        print()

print_topics(lda_model, vectorizer)

Topic 1:
호텔: 927.1300
안내: 706.1683
숙소: 533.9560
주변: 499.1181
버스: 457.1994
근처: 450.1976
관광지: 388.1988
시간: 379.1593
배차: 275.1990
확인: 200.4096

Topic 2:
예약: 2463.7588
숙박: 1425.3883
변경: 503.1972
병원: 486.6149
확인: 483.8619
시간: 425.9296
취소: 239.1981
날짜: 216.1981
숙소: 214.4416
부탁: 151.2704

Topic 3:
병원: 1361.7695
주차: 1096.1984
요금: 807.1981
얼마: 498.0012
시설: 494.3675
차비: 470.1770
지금: 394.1801
지도: 332.7317
체크아웃: 320.1948
위치: 273.2628

Topic 4:
호출: 374.9303
간병인: 352.1995
간호사: 329.1971
직원: 317.7922
여기: 243.4932
달라: 202.1983
어디: 167.8050
오라: 165.0812
이익: 142.7595
공항: 124.9092

Topic 5:
진료: 1059.2103
예약: 892.6388
체크: 706.1955
시간: 609.4329
언제: 534.0088
내과: 369.1979
병원: 360.1954
오늘: 312.9898
확인: 306.8564
담당: 291.0267



/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
